In [1]:
########### Import Libraries ###########
########################################

import os
import sys
import re
import json
import random
import subprocess
import time
import traceback

import pandas as pd

from pprint import pprint
from tabulate import tabulate

import Helpers.env
from Agents.multiAgentDS import run_ACL_workflow
from Helpers.Formats import wrap_text, wrap_dataframe, extract_table_info, parse_kv_response, csv_to_list,to_none_if_noneish, _norm_nl
from Helpers.Read_Files import read_all_files_in_folder, read_all_json_files_in_folder, read_all_vpc_files_in_folder, read_topology_file, read_all_json_files_in_folder_Eval
from Helpers.Parse import parse_config_to_json, resolve_names, ensure_topology_dict 
from Helpers.Finder import find_pcs_in_same_network, get_device_info, get_pc_name_by_ip, extract_router_facts_from_json, choose_device_only
from Helpers.RuleConflict import main_conflictDetection
from GNS3.GNS3 import apply_config_GNS3, Router_Access

from Batfish.Batfish import Initialization_Batfish_session, prepare_batfish_context
from Batfish.preprocess import *
from Batfish.preprocess import _intent_directionality, _infer_stage_from_results
from Batfish.Questions import *


from Helpers.configs import process_configuration, generate_config2_file, generate_config2_file_apply_only, add_updated_router_cmd, extract_rule_and_ACL_lines

from Agents.AgentsDS import * #client, get_direction, Answer_Query
# from deepseek_tool import *


from Evaluation.Helper import *
from Evaluation.Conf_DetectorEval import *
from Evaluation.EvaluateDS import evaluate_acl_generator_runtime, evaluate_entity_extractor, run_full_entity_eval, export_entity_eval_excel, evaluate_choose_device_only_deterministic, evaluate_query_agent_all, evaluate_acl_generator_all, append_device_eval_to_excel, evaluate_acl_generator_all_end_to_end_merged, evaluate_acl_generator_all_end_to_end#evaluate_end_to_end_aclbleu, evaluate_acl_generator_all_end_to_end


import getpass
import telnetlib
import ipaddress

from swarm import Swarm, Agent

# noinspection PyUnresolvedReferences
from pybatfish.client.session import Session
from pybatfish.datamodel import Edge, Interface  
from pybatfish.datamodel.answer import TableAnswer
from pybatfish.datamodel.flow import HeaderConstraints, PathConstraints  
from pybatfish.datamodel.route import BgpRoute  
from pybatfish.util import get_html

# %run startup.py


In [2]:
# #################### Device information to access GNS3 server: consol IP and port ####################
######################################################################################################
Consol = [  # extracted from GNS3 topology summary
    {
        "D_Name": "APP1",
        "D_IP": "10.133.14.34",
        "D_Port": "5042"
    },
    {
        "D_Name": "ISP",
        "D_IP": "10.133.14.34",
        "D_Port": "5045"
    },
    {
        "D_Name": "PC1",
        "D_IP": "10.133.14.34",
        "D_Port": "5012"
    },
    {
        "D_Name": "PC2",
        "D_IP": "10.133.14.34",
        "D_Port": "5014"
    },
    {
        "D_Name": "PC3",
        "D_IP": "10.133.14.34",
        "D_Port": "5016"
    },
    {
        "D_Name": "PC4",
        "D_IP": "10.133.14.34",
        "D_Port": "5018"
    },
    {
        "D_Name": "PC5",
        "D_IP": "10.133.14.34",
        "D_Port": "5020"
    },
    {
        "D_Name": "PC6",
        "D_IP": "10.133.14.34",
        "D_Port": "5022"
    },
    {
        "D_Name": "PC7",
        "D_IP": "10.133.14.34",
        "D_Port": "5036"
    },
    {
        "D_Name": "PC8",
        "D_IP": "10.133.14.34",
        "D_Port": "5038"
    },
    {
        "D_Name": "R1",
        "D_IP": "10.133.14.34",
        "D_Port": "5006"
    },
    {
        "D_Name": "R2",
        "D_IP": "10.133.14.34",
        "D_Port": "5007"
    },
    {
        "D_Name": "R3",
        "D_IP": "10.133.14.34",
        "D_Port": "5008"
    },
    {
        "D_Name": "WEB1",
        "D_IP": "10.133.14.34",
        "D_Port": "5040"
    }
]

Sub_Net = [  # extracted from current topology design
    {
        "D_Name": "PC1",
        "D_IP": "192.168.10.10",
        "D_Net": "192.168.10.0"
    },
    {
        "D_Name": "PC2",
        "D_IP": "192.168.10.20",
        "D_Net": "192.168.10.0"
    },
    {
        "D_Name": "PC3",
        "D_IP": "192.168.10.30",
        "D_Net": "192.168.10.0"
    },
    {
        "D_Name": "PC6",
        "D_IP": "192.168.10.40",
        "D_Net": "192.168.10.0"
    },
    {
        "D_Name": "WEB1",
        "D_IP": "192.168.20.10",
        "D_Net": "192.168.20.0"
    },
    {
        "D_Name": "APP1",
        "D_IP": "192.168.20.20",
        "D_Net": "192.168.20.0"
    },
    {
        "D_Name": "PC4",
        "D_IP": "192.168.30.10",
        "D_Net": "192.168.30.0"
    },
    {
        "D_Name": "PC5",
        "D_IP": "192.168.30.20",
        "D_Net": "192.168.30.0"
    },
    {
        "D_Name": "PC7",
        "D_IP": "192.168.30.30",
        "D_Net": "192.168.30.0"
    },
    {
        "D_Name": "PC8",
        "D_IP": "192.168.30.40",
        "D_Net": "192.168.30.0"
    }
]
#################### Device information to access GNS3 server: consol IP and port ####################
# ######################################################################################################
# Consol = [ # this info extracted from GNS3 consol (they are based on the VM server)
#     {
#         "D_Name" : "PC1",
#         "D_IP"   : "10.133.14.34",
#         "D_Port" : "5012"
#     },
#     {
#         "D_Name" : "PC2",
#         "D_IP"   : "10.133.14.34",
#         "D_Port" : "5014"
#     },
#     {
#         "D_Name" : "PC3",
#         "D_IP"   : "10.133.14.34",
#         "D_Port" : "5016"
#     },
#     {
#         "D_Name" : "PC4",
#         "D_IP"   : "10.133.14.34",
#         "D_Port" : "5018"
#     },
#     {
#         "D_Name" : "PC5",
#         "D_IP"   : "10.133.14.34",
#         "D_Port" : "5020"
#     },
#     {
#         "D_Name" : "PC6",
#         "D_IP"   : "10.133.14.34",
#         "D_Port" : "5022"
#     },
#     {
#         "D_Name" : "R1",
#         "D_IP"   : "10.133.14.34",
#         "D_Port" : "5006"
#     },
#     {
#         "D_Name" : "R2",
#         "D_IP"   : "10.133.14.34",
#         "D_Port" : "5007"
#     },
#     {
#         "D_Name" : "R3",
#         "D_IP"   : "10.133.14.34",
#         "D_Port" : "5008"
#     }
# ]

# Sub_Net = [ # this info extracted from GNS3 topology
#     {
#         "D_Name" : "PC1",
#         "D_IP"   : "172.16.10.2",
#         "D_Net" : "172.16.10.0"
#     },
#     {
#         "D_Name" : "PC2",
#         "D_IP"   : "172.16.10.3",
#         "D_Net" : "172.16.10.0"
#     },
#     {
#         "D_Name" : "PC3",
#         "D_IP"   : "172.16.30.2",
#         "D_Net" : "172.16.30.0"
#     },
#     {
#         "D_Name" : "PC4",
#         "D_IP"   : "172.16.30.3",
#         "D_Net" : "172.16.30.0"
#     },
#     {
#         "D_Name" : "PC5",
#         "D_IP"   : "172.16.50.2",
#         "D_Net" : "172.16.50.0"
#     },
#     {
#         "D_Name" : "PC6",
#         "D_IP"   : "172.16.50.3",
#         "D_Net" : "172.16.50.0"
#     }
# ]

In [3]:
############# Set input variables + default variables to the system: intent, etc ##############
################################################################################################

# parse_config_to_json('configs/R1')#.cfg')
# parse_config_to_json('configs/R2')#.cfg')
# parse_config_to_json('configs/R3')#.cfg')

# new_intent = input("Hi!\n Type your intent here : ").strip().lower()    
# new_intent = " Want R1 to deny entire 172.16.30.0 network." #wrong extraction  
                                                            #  --> ambiguous: Case 1 — Block traffic TO 172.16.30.0  
                                                            #                 Case 2 — Block traffic FROM 172.16.30.0
# new_intent = "We want R1 to permit only the Engineering workstation 172.16.50.2/24 to be able to access the web server in Administrative network with the IP address 172.16.10.2 and port address 80."
#### add lines for interfcaes without names...???
# eval fail here, LLM correct in choosing the device


# new_intent = "Permit only the host 172.16.30.2 from exiting the sales network" 
# new_intent = "Permit only the host 172.16.50.3 from exiting the engineering network" 
# new_intent = "Permit only the host 172.16.30.3 from exiting router R2" #sales network" 
# new_intent = "Prohibit SSH access to subnet 172.16.50.0/24"

# file_path           = "./configs"  # Replace with your folder path
# file_contents       = read_all_json_files_in_folder(file_path)
# network_topology            = "./configs/TopologyFile.json"

######  Used Groundtruth for Evaluation LLM Performance  ######

file_path           = "./Groundtruth/Multirouter_generated/router_configs/" #"./configs"  # Replace with your folder path
file_contents       = read_all_json_files_in_folder_Eval(file_path)
network_topology            = "./Groundtruth/multirouter_topology.json" #"./configs/TopologyFile.json"
new_intent          = "Allow PC1 to SSH to WEB1 in the DMZ"
# new_intent    = "Block PC1 from accessing the Internet on HTTPS." # no exist but give it exist??
# new_intent    = "Block PC3 from accessing the DMZ network."
# new_intent          = "Block the LAN network from sending DNS queries to the Internet."
# # # new_intent          = "Deny LAN from accessing the DMZ on HTTP."
# new_intent                = "Block PC3 from accessing the DMZ network"
Eval_GT_path             = "./Groundtruth/Multirouter_generated/multirouter_intents_mixed.json"
# Eval_GT_path             = "./Groundtruth/Multirouter_generated/multirouter_intents_mixed_sample.json"
Conflict_GT_PATH         = "./Groundtruth/acl_conflict_detection_GT.json"

######## Load Network topology #######
with open(network_topology, "r", encoding="utf-8") as f:
    network_topology_json = json.load(f)

######## Load GT for agents & modules evaluation #######
with open(Eval_GT_path, "r", encoding="utf-8") as f:
    Eval_GT_json = json.load(f)

######## Load Conflict GT for Rule conflict detection #######
with open(Conflict_GT_PATH, "r", encoding="utf-8") as f:
    Conflict_GT_json = json.load(f)
####### Load R configuration File/s For evaluation #######
# with open(Eval_conf_path, "r", encoding="utf-8") as f:
#     Eval_conf = f.read()

device_configs = {}
for dev in ["r1", "r2", "r3"]:
    path = f"./Groundtruth/Multirouter_generated/router_configs/{dev.upper()}.cfg"   # adjust naming
    with open(path, "r", encoding="utf-8") as f:
        device_configs[dev] = f.read()    
############# Initial values #############
# topology_file       = None
hostname            = None
Whole_configuration = None
extraction_result   = None
ACLname              = None
Rules               = None
direction           = None
Intf_Name           = None
user_action         = None
List_Found          = None

expected_ent        = None

context_variables = {
    # "new_intent" : new_intent,
    "file_path"  : file_path,
    # "topology_file" : file_contents,
    "network_topology" : network_topology,
    "Eval_GT_File"     : Eval_GT_json,
    "Conflict_GT_File" : Conflict_GT_json
    
}

In [ ]:
# compute runtime + cpu 
df_time, summary_time, paper_table, latex_table = evaluate_acl_generator_runtime(
    Eval_GT_json,
    sample_size=100,
    seed=42,
    out_xlsx="ACL_generator_runtime_eval_DS.xlsx",
)

[DeepSeek:deepseek-coder] Processing: 'Render Cisco IOS ACL configuration commands from these already-validated entities.\n        You MUST NOT infer anything b' ...


INFO:httpx:HTTP Request: POST https://api.deepseek.com/v1/chat/completions "HTTP/1.1 200 OK"


[DeepSeek RAW OUTPUT]: 'ip access-list extended ACL_F0_0_IN\n deny tcp host 203.0.113.1 192.168.10.0 0.0.0.255 eq 23\ninterface f0/0\n ip access-group ACL_F0_0_IN in'
[DeepSeek:deepseek-coder] Processing: 'Render Cisco IOS ACL configuration commands from these already-validated entities.\n        You MUST NOT infer anything b' ...


INFO:httpx:HTTP Request: POST https://api.deepseek.com/v1/chat/completions "HTTP/1.1 200 OK"


[DeepSeek RAW OUTPUT]: 'ip access-list extended ACL_F0_0_IN\n deny tcp host 203.0.113.1 192.168.10.0 0.0.0.255 eq 23\ninterface f0/0\n ip access-group ACL_F0_0_IN in'
[1/100] mr_mixed_0118_p3 -> ok, wall=3.2303s, cpu=0.0055s
[DeepSeek:deepseek-coder] Processing: 'Render Cisco IOS ACL configuration commands from these already-validated entities.\n        You MUST NOT infer anything b' ...


INFO:httpx:HTTP Request: POST https://api.deepseek.com/v1/chat/completions "HTTP/1.1 200 OK"


In [4]:
# (1) Evaluation: EntityExtraction Agent  
# -------------------------------------
df_per_case, overall_metrics = run_full_entity_eval(
                 Eval_GT_json = Eval_GT_json,
             network_topology = network_topology,
             output_file = "entity_extraction_eval_DeepSeek.xlsx",
             relaxed_ip_match = True,
)
print("Evaluation results Saved:", "entity_extraction_eval_DeepSeek.xlsx")
pprint(overall_metrics)

df_per_case.to_csv("entity_extraction_per_case_DeepSeek.csv", index=False)

print(df_per_case.columns.tolist())

df_per_case = export_entity_eval_excel(
                df_per_case = df_per_case,
            overall_metrics = overall_metrics,
                output_file = "entity_extraction_eval_all_DeepSeek.xlsx"
        )



ID: mr_mixed_0001
Intent: Block traffic from LAN_B to PC3 over SSH.
[DeepSeek:deepseek-chat] Processing: 'Intent:\n        Block traffic from LAN_B to PC3 over SSH.\n        \n        Resolved dictionary (GROUND TRUTH):\n        {' ...


INFO:httpx:HTTP Request: POST https://api.deepseek.com/v1/chat/completions "HTTP/1.1 200 OK"


[DeepSeek RAW OUTPUT]: 'Source IP: None\nDestination IP: 192.168.10.30\nProtocol: tcp\nPort: 22\nAction: deny\nApplication: SSH\nSource IP Subnet: 192.168.30.0/24\nDestination IP Subnet: None'

ID: mr_mixed_0001_p1
Intent: Block LAN_B from accessing PC3 on SSH.
[DeepSeek:deepseek-chat] Processing: 'Intent:\n        Block LAN_B from accessing PC3 on SSH.\n        \n        Resolved dictionary (GROUND TRUTH):\n        {\n  ' ...


INFO:httpx:HTTP Request: POST https://api.deepseek.com/v1/chat/completions "HTTP/1.1 200 OK"


[DeepSeek RAW OUTPUT]: 'Source IP: None\nDestination IP: 192.168.10.30\nProtocol: tcp\nPort: 22\nAction: deny\nApplication: None\nSource IP Subnet: 192.168.30.0/24\nDestination IP Subnet: None'

ID: mr_mixed_0001_p2
Intent: Deny LAN_B from reaching PC3 using SSH.
[DeepSeek:deepseek-chat] Processing: 'Intent:\n        Deny LAN_B from reaching PC3 using SSH.\n        \n        Resolved dictionary (GROUND TRUTH):\n        {\n ' ...


...

[DeepSeek RAW OUTPUT]: 'Source IP: None\nDestination IP: 192.168.10.30\nProtocol: None\nPort: None\nAction: deny\nApplication: DNS\nSource IP Subnet: 192.168.30.0/24\nDestination IP Subnet: None'

ID: mr_mixed_0182
Intent: Allow PC7 to reach APP1 on DNS as traffic exits the selected router.
[DeepSeek:deepseek-chat] Processing: 'Intent:\n        Allow PC7 to reach APP1 on DNS as traffic exits the selected router.\n        \n        Resolved dictionar' ...


INFO:httpx:HTTP Request: POST https://api.deepseek.com/v1/chat/completions "HTTP/1.1 200 OK"


[DeepSeek RAW OUTPUT]: 'Source IP: 192.168.30.30\nDestination IP: 192.168.20.20\nProtocol: udp\nPort: 53\nAction: permit\nApplication: DNS\nSource IP Subnet: None\nDestination IP Subnet: None'

ID: mr_mixed_0182_p1
Intent: Permit PC7 to use DNS toward APP1 at egress.
[DeepSeek:deepseek-chat] Processing: 'Intent:\n        Permit PC7 to use DNS toward APP1 at egress.\n        \n        Resolved dictionary (GROUND TRUTH):\n      ' ...


INFO:httpx:HTTP Request: POST https://api.deepseek.com/v1/chat/completions "HTTP/1.1 200 OK"


[DeepSeek RAW OUTPUT]: 'Source IP: 192.168.30.30\nDestination IP: 192.168.20.20\nProtocol: udp\nPort: 53\nAction: permit\nApplication: DNS\nSource IP Subnet: None\nDestination IP Subnet: None'

Saved results to: entity_extraction_eval_DeepSeek_lastchat_port.xlsx

================ OVERALL METRICS ================
{'confusion_action': {('deny', 'deny'): 242, ('permit', 'permit'): 258},
 'confusion_protocol': {('icmp', 'icmp'): 48,
                        ('ip', 'ip'): 24,
                        ('ip', None): 24,
                        ('tcp', 'tcp'): 330,
                        ('udp', 'udp'): 69,
                        ('udp', None): 5},
 'errors_sample': [{'field_matches': {'action': True,
                                      'dst': True,
                                      'port': False,
                                      'protocol': False,
                                      'src': True},
                    'gt_norm': {'action': 'permit',
                                

In [5]:
# (3) Evaluation: ACL Placement Resolver Agent
# ----------------------------------------------
df_q, df_q_sum = evaluate_query_agent_all(
              Eval_GT_json    = Eval_GT_json,
              topo_prompt     = network_topology_json,
              # device_configs  = device_configs,
              out_xlsx        = "Query_agent_eval_DeepSeekCoder.xlsx"
)

INFO:httpx:HTTP Request: POST https://api.deepseek.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.deepseek.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.deepseek.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.deepseek.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.deepseek.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.deepseek.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.deepseek.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.deepseek.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.deepseek.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.deepseek.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.deepseek.com/v1/chat/complet

Saved: Query_agent_eval_DeepSeekCoder.xlsx


In [4]:
# (4) Evaluation: ACL Generator Agent
# -----------------------------------
df_acl, df_acl_sum = evaluate_acl_generator_all(
                         Eval_GT_json = Eval_GT_json,
                             out_xlsx = "ACL_generator_eval_DeepSeekCoder.xlsx",
            require_full_config_match = True
)
print(df_acl_sum)

[DeepSeek:deepseek-coder] Processing: 'Render Cisco IOS ACL configuration commands from these already-validated entities.\n        You MUST NOT infer anything b' ...


INFO:httpx:HTTP Request: POST https://api.deepseek.com/v1/chat/completions "HTTP/1.1 200 OK"


[DeepSeek RAW OUTPUT]: 'ip access-list extended ACL_G0_2_IN\n deny tcp 192.168.30.0 0.0.0.255 host 192.168.10.30\ninterface g0/2\n ip access-group ACL_G0_2_IN in'
[DeepSeek:deepseek-coder] Processing: 'Render Cisco IOS ACL configuration commands from these already-validated entities.\n        You MUST NOT infer anything b' ...


INFO:httpx:HTTP Request: POST https://api.deepseek.com/v1/chat/completions "HTTP/1.1 200 OK"


[DeepSeek RAW OUTPUT]: 'ip access-list extended ACL_G0_2_IN\n deny tcp 192.168.30.0 0.0.0.255 host 192.168.10.30\ninterface g0/2\n ip access-group ACL_G0_2_IN in'
[DeepSeek:deepseek-coder] Processing: 'Render Cisco IOS ACL configuration commands from these already-validated entities.\n        You MUST NOT infer anything b' ...


...

[DeepSeek RAW OUTPUT]: 'ip access-list extended ACL_G0_0_OUT\n deny udp 192.168.30.0 0.0.0.255 host 192.168.10.30\ninterface g0/0\n ip access-group ACL_G0_0_OUT out'
[DeepSeek:deepseek-coder] Processing: 'Render Cisco IOS ACL configuration commands from these already-validated entities.\n        You MUST NOT infer anything b' ...


INFO:httpx:HTTP Request: POST https://api.deepseek.com/v1/chat/completions "HTTP/1.1 200 OK"


[DeepSeek RAW OUTPUT]: 'ip access-list extended ACL_G0_1_OUT\n permit udp host 192.168.30.30 host 192.168.20.20\ninterface g0/1\n ip access-group ACL_G0_1_OUT out'
[DeepSeek:deepseek-coder] Processing: 'Render Cisco IOS ACL configuration commands from these already-validated entities.\n        You MUST NOT infer anything b' ...


INFO:httpx:HTTP Request: POST https://api.deepseek.com/v1/chat/completions "HTTP/1.1 200 OK"


[DeepSeek RAW OUTPUT]: 'ip access-list extended ACL_G0_1_OUT\n permit udp host 192.168.30.30 host 192.168.20.20\ninterface g0/1\n ip access-group ACL_G0_1_OUT out'
Saved: ACL_generator_eval_DeepSeekCoder_port.xlsx
     n  acc_rule_semantics  acc_interface_binding  acc_full_config_after_rename  record_accuracy
0  500               0.192                    1.0                         0.192            0.192


In [5]:
df_aclE, df_acl_sumE = evaluate_acl_generator_all_end_to_end(
                         Eval_GT_json = Eval_GT_json,
                        context_variables = context_variables,
                        out_xlsx = "ACL_generator_eval_end-to_end_Accuracy_DeepSeekCoder_port.xlsx",
            require_full_config_match = True
)

# df_e2e_bleu, summary_e2e_bleu = evaluate_end_to_end_aclbleu(
#     Eval_GT_json=Eval_GT_json,
#     base_context_variables=context_variables,
#     topology_dict=context_variables.get("objects", None),   # or your topology dict
#     out_xlsx="end_to_end_aclbleu.xlsx",
#     alpha=0.1,
#     beta=0.2,
#     gamma=0.3,
#     delta=0.4,
#     aclbleu_threshold=0.90
# )

new_intent = 'Block traffic from LAN_B to PC3 over SSH.'
[DeepSeek:deepseek-chat] Processing: 'Intent:\n        Block traffic from LAN_B to PC3 over SSH.\n        \n        Resolved dictionary (GROUND TRUTH):\n        {' ...


INFO:httpx:HTTP Request: POST https://api.deepseek.com/v1/chat/completions "HTTP/1.1 200 OK"


[DeepSeek RAW OUTPUT]: 'Source IP: None\nDestination IP: 192.168.10.30\nProtocol: tcp\nPort: 22\nAction: deny\nApplication: SSH\nSource IP Subnet: 192.168.30.0/24\nDestination IP Subnet: None'
extraction_result: Source IP: None
Destination IP: 192.168.10.30
Protocol: tcp
Port: 22
Action: deny
Application: SSH
Source IP Subnet: 192.168.30.0/24
Destination IP Subnet: None
interface_map sample keys: ['R1', 'R2', 'R3']
interface_map sample value: {'g0/0': {'role': 'LAN_A', 'ip': '192.168.10.1/24', 'connected_to': 'SW1'}, 'g0/1': {'role': 'TRANSIT_TO_R2', 'ip': '10.0.12.1/30', 'connected_to': 'R2:g0/0'}}
src_token 192.168.30.0/24 -> 192.168.30.0/24
dst_token 192.168.10.30 -> 192.168.10.30
edge R2
File_name
R1
[DeepSeek:deepseek-chat] Processing: 'Topology (JSON):\n        {\n  "objects": {\n    "LAN_A": "192.168.10.0/24",\n    "PC1": "192.168.10.10/32",\n    "PC2": "192' ...


INFO:httpx:HTTP Request: POST https://api.deepseek.com/v1/chat/completions "HTTP/1.1 200 OK"


[DeepSeek RAW OUTPUT]: 'None'
[DeepSeek:deepseek-chat] Processing: 'Intent:\n        Block traffic from LAN_B to PC3 over SSH.\n        \n        Chosen device:\n        R1\n        \n        In' ...


...

[DeepSeek RAW OUTPUT]: 'out'
[DeepSeek:deepseek-chat] Processing: 'Router configuration:\n        ip access-list extended ACL_R3_G0_0_OUT\n 50 deny icmp host 192.168.20.20 any\n 140 deny tcp' ...


INFO:httpx:HTTP Request: POST https://api.deepseek.com/v1/chat/completions "HTTP/1.1 200 OK"


[DeepSeek RAW OUTPUT]: 'ACL_R3_G0_2_OUT'
acl_name_raw: 'ACL_R3_G0_2_OUT'
[DeepSeek:deepseek-coder] Processing: 'Render Cisco IOS ACL configuration commands from these already-validated entities.\n        You MUST NOT infer anything b' ...


INFO:httpx:HTTP Request: POST https://api.deepseek.com/v1/chat/completions "HTTP/1.1 200 OK"


[DeepSeek RAW OUTPUT]: 'ip access-list extended ACL_R3_G0_2_OUT\n permit udp host 192.168.30.30 host 192.168.20.20 eq 53\ninterface g0/2\n ip access-group ACL_R3_G0_2_OUT out'
generated
ACLRule(acl_name='ACL_R3_G0_2_OUT', sequence=9999, action='permit', protocol='udp', src='192.168.30.30/32', dst='192.168.20.20/32', src_port=None, dst_port=53, router='R3', interface='g0/2', direction='out', raw_line='permit udp host 192.168.30.30 host 192.168.20.20 eq 53')
interface_name in add_updated_router_cmd
g0/2
Saved: ACL_generator_eval_end-to_end_Accuracy_DeepSeekCoder_port.xlsx
